# World Models 2: Training and Exploring the Visual VAE

## Case study: compressing CarRacing observations

The world model first needs a compact visual state. A convolutional variational autoencoder maps a 64×64 RGB frame to a Gaussian latent distribution and reconstructs it. The latent representation will become the input to the recurrent dynamics model.

### VAE theory

The encoder estimates $q_\phi(z\mid x)=\mathcal N(\mu,\mathrm{diag}(\sigma^2))$. The reparameterization trick samples $z=\mu+\sigma\odot\epsilon$, where $\epsilon\sim\mathcal N(0,I)$, so gradients can pass through sampling. The decoder maps $z$ back to an image. Training minimizes the negative ELBO: reconstruction error plus $\beta D_{KL}(q_\phi(z\mid x)\Vert p(z))$. Reconstruction preserves detail; KL regularization encourages a smooth, organized latent space.

A compact code can summarize road geometry and car position, reducing the cost of learning dynamics from pixels. However, good reconstructions do not prove that the latent retains every feature needed for control. This notebook reuses notebook 01's rollout file or creates procedural frames if it is absent. The fallback is not real CarRacing data.

## 1. Load frames and split data

Frames are normalized to `[0,1]` and converted to channel-first tensors. We use a deterministic held-out split. When no rollout file exists, varied procedural road scenes keep the example self-contained.

**What this cell does:** loads saved observations or creates fallback images, then prepares train and validation tensors.

In [ ]:
import random
from pathlib import Path
import numpy as np, torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
SEED=41; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu"); ROOT=Path("world-model-data"); ROOT.mkdir(exist_ok=True)
DATA=ROOT/"car_racing_rollouts.npz"; ZDIM=32; SIZE=64

def demo_frames(n=800):
 out=[]
 for _ in range(n):
  im=np.zeros((SIZE,SIZE,3),np.uint8); im[:,:,1]=75; im[:,:,0]=30; im[:,:,2]=25
  y=np.linspace(0,1,SIZE); p=np.random.uniform(-1,1); c=np.random.uniform(-1,1); center=32+(y-.2)*c*23-p*12; half=5+y*25
  for r in range(SIZE):
   l=max(0,int(center[r]-half[r])); q=min(SIZE,int(center[r]+half[r])); im[r,l:q]=[78,78,78]
   if l>0:im[r,max(0,l-2):l]=[225,225,210]
   if q<SIZE:im[r,q:min(SIZE,q+2)]=[225,225,210]
  x=int(np.clip(32+p*13,5,58)); im[51:57,x-4:x+4]=[220,50,45]; out.append(im)
 return np.stack(out)
if DATA.exists():
 d=np.load(DATA); images=d['observations']; print("Loaded rollout images:",len(images))
else: images=demo_frames(); print("Using procedural images:",len(images))
perm=np.random.permutation(len(images)); cut=int(.9*len(images))
train=torch.tensor(images[perm[:cut]],dtype=torch.float32).permute(0,3,1,2)/255
valid=torch.tensor(images[perm[cut:]],dtype=torch.float32).permute(0,3,1,2)/255
print(train.shape,valid.shape,"device",device)


## 2. Convolutional VAE architecture

Strided convolutions reduce spatial resolution; two heads produce the latent mean and log-variance. The decoder upsamples the latent vector back to the image size. The sampling operation uses the differentiable reparameterization trick.

**What this cell does:** defines the encoder, latent sampler, decoder, and checks tensor dimensions.

In [ ]:
class ConvVAE(nn.Module):
 def __init__(self,zdim=ZDIM):
  super().__init__(); self.enc=nn.Sequential(nn.Conv2d(3,32,4,2,1),nn.ReLU(),nn.Conv2d(32,64,4,2,1),nn.ReLU(),nn.Conv2d(64,128,4,2,1),nn.ReLU(),nn.Flatten())
  self.mu=nn.Linear(8192,zdim); self.lv=nn.Linear(8192,zdim); self.fc=nn.Linear(zdim,8192)
  self.dec=nn.Sequential(nn.Unflatten(1,(128,8,8)),nn.ConvTranspose2d(128,64,4,2,1),nn.ReLU(),nn.ConvTranspose2d(64,32,4,2,1),nn.ReLU(),nn.ConvTranspose2d(32,3,4,2,1),nn.Sigmoid())
 def encode(self,x):h=self.enc(x);return self.mu(h),self.lv(h).clamp(-10,8)
 def decode(self,z):return self.dec(self.fc(z))
 def forward(self,x):
  m,l=self.encode(x); z=m+torch.exp(.5*l)*torch.randn_like(m); return self.decode(z),m,l
vae=ConvVAE().to(device); print("parameters:",sum(p.numel() for p in vae.parameters()),"output:",vae(train[:2].to(device))[0].shape)


## 3. ELBO objective and short training

Binary cross-entropy measures pixel reconstruction; the KL term penalizes divergence from a unit Gaussian prior. We use a small beta and a short schedule to keep this demonstration quick. The loss values depend on image resolution and reduction convention, so compare only like-for-like runs.

**What this cell does:** trains the VAE, logs reconstruction and KL components, and plots training and validation objectives.

In [ ]:
BATCH=64; EPOCHS=7; BETA=.15; opt=torch.optim.Adam(vae.parameters(),lr=1e-3)
def loss_fn(x,r,m,l):
 rec=F.binary_cross_entropy(r,x,reduction='sum')/len(x); kl=-.5*(1+l-m.square()-l.exp()).sum()/len(x); return rec+BETA*kl,rec,kl
hist=[]; vhist=[]
for ep in range(EPOCHS):
 vae.train(); vals=[]
 for ix in torch.randperm(len(train)).split(BATCH):
  x=train[ix].to(device); r,m,l=vae(x); loss,rec,kl=loss_fn(x,r,m,l)
  opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); vals.append([loss.item(),rec.item(),kl.item()])
 vae.eval()
 with torch.no_grad():
  x=valid[:min(BATCH,len(valid))].to(device); vl=loss_fn(x,*vae(x))[0].item()
 hist.append(np.mean(vals,axis=0)); vhist.append(vl)
 print(f"epoch {ep+1}/{EPOCHS}: total {hist[-1][0]:.1f}, recon {hist[-1][1]:.1f}, KL {hist[-1][2]:.1f}, val {vl:.1f}")
plt.plot([v[0] for v in hist],label='train');plt.plot(vhist,label='validation');plt.xlabel('epoch');plt.ylabel('ELBO loss');plt.grid();plt.legend();plt.show()


## 4. Reconstruction and latent interpolation

We compare inputs with reconstructions, then interpolate between two posterior means. Smooth changes suggest that nearby latent points decode to related images, although straight lines are not guaranteed to remain on the data manifold.

**What this cell does:** visualizes reconstruction quality and decodes a short interpolation path.

In [ ]:
vae.eval()
with torch.no_grad():
 x=valid[:8].to(device); r,m,l=vae(x)
fig,ax=plt.subplots(2,8,figsize=(14,4))
for i in range(8):
 ax[0,i].imshow(x[i].cpu().permute(1,2,0));ax[0,i].axis('off');ax[0,i].set_title('input')
 ax[1,i].imshow(r[i].cpu().permute(1,2,0));ax[1,i].axis('off');ax[1,i].set_title('recon')
plt.tight_layout();plt.show()
with torch.no_grad():
 path=torch.stack([(1-a)*m[0]+a*m[1] for a in torch.linspace(0,1,8,device=device)]); dec=vae.decode(path).cpu()
fig,ax=plt.subplots(1,8,figsize=(14,2))
for i,a in enumerate(ax):a.imshow(dec[i].permute(1,2,0));a.axis('off')
plt.show()


## 5. Export latents for dynamics training

We encode frames with the posterior mean, giving deterministic latent sequences. The checkpoint and aligned arrays are saved under `world-model-data/` for notebook 03. Keeping episode IDs prevents false transitions across resets.

**What this cell does:** saves the VAE and posterior means with the corresponding action, reward, done, and episode arrays.

In [ ]:
@torch.no_grad()
def encode(arr):
 vae.eval(); out=[]
 for st in range(0,len(arr),128):
  x=torch.tensor(arr[st:st+128],dtype=torch.float32).permute(0,3,1,2).to(device)/255; m,_=vae.encode(x);out.append(m.cpu().numpy())
 return np.concatenate(out)
latents=encode(images); ckpt=ROOT/'visual_vae.pt'; target=ROOT/'latent_rollouts.npz'
torch.save({'state_dict':vae.state_dict(),'latent_dim':ZDIM},ckpt)
if DATA.exists():
 d=np.load(DATA); actions,rewards,dones,episode_ids=(d[k] for k in ('actions','rewards','dones','episode_ids'))
else:
 actions=np.random.uniform(-1,1,(len(latents),3)).astype(np.float32); rewards=np.zeros(len(latents),np.float32); dones=np.zeros(len(latents),np.float32); episode_ids=np.arange(len(latents))//40
np.savez_compressed(target,latents=latents,actions=actions,rewards=rewards,dones=dones,episode_ids=episode_ids)
print("saved",ckpt,"latent shape",latents.shape,"rollout",target)


## Key takeaways

- A VAE maps images to a Gaussian latent distribution and learns reconstruction through the ELBO.
- Reparameterization enables gradient-based learning through stochastic sampling.
- Reconstruction fidelity and latent regularity trade off against each other.
- Posterior means provide deterministic latent trajectories for dynamics training.
- Visual quality alone does not prove that a latent representation is sufficient for control.
- Next, the MDN-RNN learns action-conditioned latent transitions, rewards, and termination.